# Notebook 03 — Automatic Clustering
## Best Case vs. Worst Case

**Key Insight**: When your queries don't align with natural load order, a cluster key reorganizes partitions to match your access pattern.

| | Best Design | Worst Design |
|---|---|---|
| **Cluster key** | `account_id` (matches customer lookup pattern) | No clustering on lookup column |
| **After reclustering** | 95%+ partition pruning | Full table scan |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

-- Disable result cache for accurate performance comparisons
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

---
## Step 1: The Problem — account_id is Scattered

The `TRANSACTIONS` table is loaded chronologically. Great for date-range queries (Notebook 02).  
But `account_id` is distributed across ALL partitions — customer lookups force a full scan.

In [ ]:
-- How well is TRANSACTIONS clustered on account_id? (expect POOR — high depth)
SELECT SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS_UNORDERED', '(account_id)');

In [ ]:
-- WORST CASE: account_id lookup on unordered table = full scan
SELECT transaction_date, merchant_name, amount, channel
FROM TRANSACTIONS_UNORDERED
WHERE account_id = 'ACC-0000012345'
ORDER BY transaction_date DESC
LIMIT 50;

**Check Query Profile** → Partitions scanned ≈ total partitions (full scan)

---
## Step 2: Create a Properly Clustered Table

We generate the same 500M rows but sorted by `account_id` at write time — producing tight non-overlapping micro-partition ranges.

In [ ]:
-- Use XL for creating 500M rows
USE WAREHOUSE HOL_XL;

-- Generate same data, but ORDERED BY account_id (simulates what Automatic Clustering achieves)
CREATE OR REPLACE TABLE TRANSACTIONS_CLUSTERED
    CLUSTER BY (account_id)
AS
SELECT
    'TXN-' || LPAD(ROW_NUMBER() OVER (ORDER BY account_id, ts), 12, '0') AS transaction_id,
    account_id,
    ts AS transaction_date,
    DATEADD(day, 1, ts)::DATE AS post_date,
    merchant_name,
    merchant_category_code,
    transaction_type,
    amount,
    currency_code,
    channel,
    is_international,
    fraud_flag,
    fraud_score
FROM (
    SELECT
        'ACC-' || LPAD(UNIFORM(0, 24999999, RANDOM())::INT, 10, '0') AS account_id,
        DATEADD(second, (SEQ8() * 0.19)::INT, '2022-01-01'::TIMESTAMP_NTZ) AS ts,
        CASE MOD(SEQ8(), 20)
            WHEN 0 THEN 'AMAZON' WHEN 1 THEN 'WALMART' WHEN 2 THEN 'TARGET'
            WHEN 3 THEN 'STARBUCKS' WHEN 4 THEN 'SHELL GAS' WHEN 5 THEN 'UBER'
            WHEN 6 THEN 'NETFLIX' WHEN 7 THEN 'COSTCO' WHEN 8 THEN 'HOME DEPOT'
            WHEN 9 THEN 'CHASE PAYMENT' WHEN 10 THEN 'VENMO TRANSFER'
            WHEN 11 THEN 'ATM WITHDRAWAL' WHEN 12 THEN 'APPLE' WHEN 13 THEN 'WHOLE FOODS'
            WHEN 14 THEN 'CVS PHARMACY' WHEN 15 THEN 'WESTERN UNION'
            WHEN 16 THEN 'CRYPTO EXCHANGE' WHEN 17 THEN 'AIRLINE TICKETS'
            WHEN 18 THEN 'HOTEL BOOKING' ELSE 'MISC RETAIL' END AS merchant_name,
        LPAD(MOD(SEQ8(), 100)::VARCHAR, 4, '0') AS merchant_category_code,
        CASE MOD(SEQ8(), 6)
            WHEN 0 THEN 'purchase' WHEN 1 THEN 'payment' WHEN 2 THEN 'transfer'
            WHEN 3 THEN 'ATM' WHEN 4 THEN 'fee' ELSE 'refund' END AS transaction_type,
        ROUND(ABS(UNIFORM(1, 50000, RANDOM())::NUMBER(12,2)) / 100, 2) AS amount,
        CASE WHEN MOD(SEQ8(), 50) = 0 THEN 'GBP'
             WHEN MOD(SEQ8(), 80) = 0 THEN 'EUR' ELSE 'USD' END AS currency_code,
        CASE MOD(SEQ8(), 5)
            WHEN 0 THEN 'branch' WHEN 1 THEN 'online' WHEN 2 THEN 'mobile'
            WHEN 3 THEN 'ATM' ELSE 'POS' END AS channel,
        MOD(SEQ8(), 50) = 0 AS is_international,
        MOD(SEQ8(), 200) = 0 AS fraud_flag,
        ROUND(UNIFORM(0, 10000, RANDOM()) / 10000.0, 4) AS fraud_score
    FROM TABLE(GENERATOR(ROWCOUNT => 500000000))
)
ORDER BY account_id;

In [ ]:
-- Switch back to XS for query testing
USE WAREHOUSE HOL_XS;

In [ ]:
-- Verify: clustering depth should be 1.0 (perfect — non-overlapping partitions)
SELECT SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS_CLUSTERED', '(account_id)');

**Expected**: `average_depth = 1.0` — each partition covers a unique, non-overlapping range of `account_id` values.

---
## Step 3: Best Case — Query the Clustered Table

In [ ]:
-- BEST CASE: Same query on clustered table (massive partition pruning)
SELECT transaction_date, merchant_name, amount, channel
FROM TRANSACTIONS_CLUSTERED
WHERE account_id = 'ACC-0000012345'
ORDER BY transaction_date DESC
LIMIT 50;

**Check Query Profile** → Compare:
- **Unordered table**: ~100% partitions scanned (full scan)
- **Clustered table**: 1-2 partitions scanned out of hundreds (99%+ pruning)

---
## Step 4: Side-by-Side Metrics

In [ ]:
-- Compare the two queries
SELECT
    CASE WHEN query_text ILIKE '%TRANSACTIONS_CLUSTERED%' THEN 'CLUSTERED'
         ELSE 'UNORDERED (full scan)' END AS version,
    total_elapsed_time / 1000 AS elapsed_sec,
    query_id,
    bytes_scanned / (1024*1024*1024) AS gb_scanned
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -30, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 50
))
WHERE query_text ILIKE '%ACC-0000012345%'
  AND query_type = 'SELECT'
  AND query_text NOT ILIKE '%QUERY_HISTORY%'
  AND query_text NOT ILIKE '%CLUSTERING_INFORMATION%'
ORDER BY start_time DESC
LIMIT 2;

---
## Step 5: Worst Case — Clustering on Low-Cardinality Column

In [ ]:
-- currency_code has only 3 values — already naturally well-clustered
SELECT SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS', '(currency_code)');

**Takeaway**: Low-cardinality columns are already well-clustered naturally. Adding a cluster key = wasted credits.

---
## Step 6: Cost Estimation

In [ ]:
-- Estimated ongoing cost to maintain clustering
SELECT SYSTEM$ESTIMATE_AUTOMATIC_CLUSTERING_COSTS('TRANSACTIONS_UNORDERED', '(account_id)');

---
## Key Takeaways

| | Best Design | Worst Design |
|---|---|---|
| **Cluster key** | High-cardinality column you filter on (`account_id`) | Low-cardinality column (`currency_code`) |
| **Result** | 99%+ pruning, 10-50x faster | Zero improvement, wasted credits |
| **When to use** | Queries don't align with natural load order | NEVER cluster on few-value columns |

**Decision framework**:
1. Check natural clustering first (`SYSTEM$CLUSTERING_INFORMATION`)
2. If already good → don't add a cluster key (free performance)
3. If poor AND you query that column frequently → add cluster key
4. Estimate cost before committing (`SYSTEM$ESTIMATE_AUTOMATIC_CLUSTERING_COSTS`)

**Next** → [Notebook 04 — Warehouse Optimization](./Notebook_04_Warehouse_Optimization.ipynb)